# 🔤 Tokenizers y Mecanismos de Atención
### UAG — Deep Learning · Semana 2 · Lunes (2h)

> **Pregunta de arranque**: ¿Cuánto "pesa" una palabra para un LLM? ¿Es lo mismo "gato" que "gatos"?

En esta sesión vas a ver cómo los modelos de lenguaje **nunca ven texto** — ven números. Entender este paso es entender el 80% de lo que hace que un Transformer funcione.


## 0. Imports y configuración

In [1]:
from transformers import AutoTokenizer

from helpers.viz import plot_tokens_colored, plot_token_comparison

## 1. ¿Qué es un token?

Antes de cargar un modelo, necesitamos entender **cómo convierte texto en números**.

Un **tokenizer** divide el texto en unidades llamadas *tokens*. Estos tokens no son necesariamente palabras: pueden ser sílabas, partes de palabras, o incluso caracteres individuales.

El proceso completo es:

```
Texto  →  [Tokenizer]  →  IDs numéricos  →  [Embedding layer]  →  Vectores  →  [Transformer]
```

> **¿Por qué no usar palabras completas?** Un vocabulario de todas las palabras en todos los idiomas sería enorme e impráctico. Los tokenizers modernos usan algoritmos como **BPE** (Byte-Pair Encoding) o **SentencePiece** para encontrar sub-palabras frecuentes.


## 2. Cargar el Tokenizer de Gemma 2

Usaremos el tokenizer oficial de Google para Gemma 2. Este es el mismo tokenizer que usa Gemma 4 internamente.

> **Nota**: Requiere haber aceptado los términos en [huggingface.co/google/gemma-2-2b](https://huggingface.co/google/gemma-2-2b) y haber corrido `hf auth login`.


In [2]:
MODEL_ID = "google/gemma-2-2b"

print(f"Cargando tokenizer de {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print(f"✅ Tokenizer listo")
print(f"   Vocabulario: {tokenizer.vocab_size:,} tokens")
print(f"   Algoritmo  : {type(tokenizer).__name__}")
print(f"   Token BOS  : {tokenizer.bos_token!r} (ID: {tokenizer.bos_token_id})")
print(f"   Token EOS  : {tokenizer.eos_token!r} (ID: {tokenizer.eos_token_id})")

Cargando tokenizer de google/gemma-2-2b...
✅ Tokenizer listo
   Vocabulario: 256,000 tokens
   Algoritmo  : GemmaTokenizer
   Token BOS  : '<bos>' (ID: 2)
   Token EOS  : '<eos>' (ID: 1)


## 3. Tu primera tokenización

Vamos a tokenizar una frase sencilla y ver exactamente qué produce el tokenizer.


In [3]:
FRASE = "A poco sí muy tokenizer"

# Tokenizar
encoded = tokenizer(FRASE, return_tensors=None)

ids     = encoded["input_ids"]
tokens  = tokenizer.convert_ids_to_tokens(ids)

print(f"Texto original : {FRASE!r}")
print(f"Número de IDs  : {len(ids)}")
print()
print(f"{'#':<5} {'Token':<20} {'ID'}")
print("-" * 35)
for i, (tok, id_) in enumerate(zip(tokens, ids)):
    print(f"{i:<5} {tok!r:<20} {id_}")


Texto original : 'A poco sí muy tokenizer'
Número de IDs  : 6

#     Token                ID
-----------------------------------
0     '<bos>'              2
1     'A'                  235280
2     '▁poco'              11975
3     '▁sí'                13376
4     '▁muy'               7495
5     '▁tokenizer'         142224


## 4. Visualización de tokens 🎨

El símbolo `▁` (underscore especial) indica que el token empieza con un espacio — es la marca de SentencePiece para separar palabras.


In [4]:
fig = plot_tokens_colored(tokens, title=f"Tokens de Gemma 2: {FRASE[:40]}...")
fig.show()

## 5. Experimento: ¿Importa el idioma?

Los LLMs modernos son multilingues, pero no todos los idiomas "cuestan" lo mismo en tokens. Vamos a compararlo.

> **Hipótesis**: Un texto en inglés tendrá menos tokens que el mismo texto en español (porque los modelos se entrenaron con más datos en inglés). ¿Será cierto?


In [5]:
textos_para_comparar = [
    "The quick brown fox jumps over the lazy dog.",      # inglés
    "El rápido zorro marrón salta sobre el perro perezoso.",  # español
    "Der schnelle braune Fuchs springt über den faulen Hund.",  # alemán
    "Le rapide renard brun saute par-dessus le chien paresseux.",  # francés
]

print("Texto → Número de tokens (Gemma 2):")
print("-" * 65)
for texto in textos_para_comparar:
    ids = tokenizer(texto)["input_ids"]
    print(f"  [{len(ids):>3} tokens] {texto}")

print()
print("💡 Observación: idiomas con menos datos de entrenamiento = más tokens por oración.")


Texto → Número de tokens (Gemma 2):
-----------------------------------------------------------------
  [ 11 tokens] The quick brown fox jumps over the lazy dog.
  [ 12 tokens] El rápido zorro marrón salta sobre el perro perezoso.
  [ 14 tokens] Der schnelle braune Fuchs springt über den faulen Hund.
  [ 16 tokens] Le rapide renard brun saute par-dessus le chien paresseux.

💡 Observación: idiomas con menos datos de entrenamiento = más tokens por oración.


## 6. Visualización comparativa: Gemma 2 vs GPT-2

Comparemos dos tokenizers de arquitecturas diferentes para el mismo texto.


In [6]:
from transformers import AutoTokenizer

tokenizer_gpt2 = AutoTokenizer.from_pretrained("gpt2")  # modelo público, sin login

textos = [
    "Machine learning",
    "Aprendizaje automático",
    "def suma(a, b): return a + b",
    "¿Cuántos planetas hay en el sistema solar?",
]

fig = plot_token_comparison(
    textos,
    tokenizers={
        "Gemma 2 (SentencePiece)": tokenizer,
        "GPT-2 (BPE)": tokenizer_gpt2,
    },
    title="Gemma 2 vs GPT-2: ¿Quién es más eficiente?",
)
fig.show()


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

c:\Users\efrai\OneDrive\Desktop\EF\Education\Professor\DL classes\uag-deep-learning\venv\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning:

`huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\efrai\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development



tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## 7. Del token al texto: decodificación

El proceso inverso: tomar IDs numéricos y reconstruir el texto original.


In [7]:
ids_ejemplo = [2, 17534, 236, 611, 168236, 25, 27, 236, 214]

# Decodificación
texto_reconstruido = tokenizer.decode(ids_ejemplo, skip_special_tokens=True)
tokens_raw = tokenizer.convert_ids_to_tokens(ids_ejemplo)

print("IDs de entrada:", ids_ejemplo)
print("Tokens crudos :", tokens_raw)
print("Texto final   :", repr(texto_reconstruido))
print()
print("✅ El tokenizer es completamente reversible — sin pérdida de información.")

IDs de entrada: [2, 17534, 236, 611, 168236, 25, 27, 236, 214]
Tokens crudos : ['<bos>', 'hello', '<0x13>', '▁on', '中止', '<unused18>', '<unused20>', '<0x13>', '</sub>']
Texto final   : 'hello\x13 on中止<unused18><unused20>\x13</sub>'

✅ El tokenizer es completamente reversible — sin pérdida de información.


## 8. Tokens especiales: el vocabulario oculto

Cada tokenizer tiene tokens que **no son palabras** pero controlan el comportamiento del modelo.


In [8]:
print("Tokens especiales de Gemma 2:")
print("-" * 50)
for nombre, token in tokenizer.special_tokens_map.items():
    if isinstance(token, str):
        id_ = tokenizer.convert_tokens_to_ids(token)
        print(f"  {nombre:<25} {token!r:<15} (ID: {id_})")
    elif isinstance(token, list):
        for t in token:
            id_ = tokenizer.convert_tokens_to_ids(t)
            print(f"  {nombre:<25} {t!r:<15} (ID: {id_})")

print()
print("💡 El token BOS (Beginning of Sequence) se inserta automáticamente al inicio.")
print("   Significa: 'aquí empieza una conversación nueva'.")


Tokens especiales de Gemma 2:
--------------------------------------------------
  bos_token                 '<bos>'         (ID: 2)
  eos_token                 '<eos>'         (ID: 1)
  unk_token                 '<unk>'         (ID: 3)
  pad_token                 '<pad>'         (ID: 0)
  mask_token                '<mask>'        (ID: 4)

💡 El token BOS (Beginning of Sequence) se inserta automáticamente al inicio.
   Significa: 'aquí empieza una conversación nueva'.


## 9. Mecanismo de Atención: ¿cómo se "miran" los tokens?

Hasta ahora vimos **cómo se codifica** el texto. Ahora el paso siguiente: dentro del Transformer, cada token "mira" a los demás para entender el contexto.

> **Intuición**: La palabra "banco" en "Me senté en el banco del parque" vs "Fui al banco a sacar dinero" tiene significados distintos. El mecanismo de atención permite al modelo distinguirlos mirando los tokens de contexto.

### ¿Qué es la atención?

Cada token produce tres vectores:
- **Query (Q)**: "¿qué estoy buscando?"
- **Key (K)**: "¿qué información ofrezco?"
- **Value (V)**: "¿qué información comparto si me seleccionan?"

La atención se calcula como:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Para visualizarlo, necesitamos cargar el modelo completo con `output_attentions=True`. Por el tamaño de Gemma 2 (2B parámetros), usaremos **DistilBERT** (66M parámetros) como demostración — el mecanismo es idéntico.


In [9]:
import torch
from transformers import AutoModel, AutoTokenizer as AutoTok

# DistilBERT: modelo compacto, mismo mecanismo de atención que Gemma
BERT_MODEL = "distilbert-base-uncased"

print(f"Cargando {BERT_MODEL}...")
bert_tokenizer = AutoTok.from_pretrained(BERT_MODEL)
bert_model = AutoModel.from_pretrained(BERT_MODEL, output_attentions=True)
bert_model.eval()
print("✅ Modelo listo")


Cargando distilbert-base-uncased...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

c:\Users\efrai\OneDrive\Desktop\EF\Education\Professor\DL classes\uag-deep-learning\venv\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning:

`huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\efrai\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development



tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Modelo listo


### 9.1 Visualizar la atención con BertViz

`bertviz` es la herramienta estándar de la industria para inspeccionar la atención interna de Transformers.


In [10]:
from bertviz import head_view, model_view

In [11]:
# Tokenizar y hacer forward pass con atenciones
FRASE_ATENCION = "El banco estaba lleno de dinero y de gente sentada."

inputs = bert_tokenizer(FRASE_ATENCION, return_tensors="pt")
bert_tokens = bert_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

with torch.no_grad():
    outputs = bert_model(**inputs)

# outputs.attentions → lista de capas → cada capa: (batch, heads, seq, seq)
atenciones = outputs.attentions

print(f"Número de capas : {len(atenciones)}")
print(f"Por capa        : (batch={atenciones[0].shape[0]}, heads={atenciones[0].shape[1]}, "
      f"seq={atenciones[0].shape[2]}, seq={atenciones[0].shape[3]})")
print(f"Tokens          : {bert_tokens}")


Número de capas : 6
Por capa        : (batch=1, heads=12, seq=18, seq=18)
Tokens          : ['[CLS]', 'el', 'banco', 'est', '##aba', 'll', '##eno', 'de', 'diner', '##o', 'y', 'de', 'gen', '##te', 'sent', '##ada', '.', '[SEP]']


In [12]:
from helpers.viz import plot_attention_heatmap
import numpy as np

# Visualizar la capa 4, cabeza 3 (suele capturar relaciones sintácticas interesantes)
CAPA = 4
CABEZA = 3

attn_matrix = atenciones[CAPA][0, CABEZA].numpy()  # (seq, seq)

fig = plot_attention_heatmap(
    attn_matrix,
    bert_tokens,
    title=f"Atención en DistilBERT — '{FRASE_ATENCION}'",
    layer=CAPA,
    head=CABEZA,
)
fig.show()


### 9.2 BertViz interactivo (si está instalado)

`bertviz` ofrece una visualización interactiva más rica, donde puedes cambiar de capa y cabeza en tiempo real.

> **Nota para clase**: Este bloque puede tomarse tiempo en renderizar. Ejecutar y esperar.


In [13]:
head_view(atenciones, bert_tokens)

<IPython.core.display.Javascript object>

## 10. ¿Qué aprendimos hoy?

| Concepto | Resumen |
|----------|---------|
| **Token** | Unidad mínima de texto que procesa un LLM. No es necesariamente una palabra. |
| **Tokenizer (SentencePiece/BPE)** | Algoritmo que divide texto en tokens balanceando vocabulario y eficiencia. |
| **Vocabulario** | Gemma 2 tiene ~256K tokens. GPT-2 tiene ~50K. |
| **Idioma y tokens** | Los idiomas con más datos de entrenamiento necesitan menos tokens por oración. |
| **Tokens especiales** | BOS, EOS, PAD → controlan el flujo del modelo, no son "contenido". |
| **Mecanismo de atención** | Cada token "mira" a todos los demás para entender el contexto. Múltiples cabezas = múltiples perspectivas simultáneas. |